# III - Stratified Random Sampling

Stratified random sampling divides a study area into distinct, non-overlapping subregions called strata, often based on characteristics relevant to the mapping (i.e. map classes) or area estimation goal (e.g., land cover type, forest change, etc). A random sample is then independently drawn from each stratum. This approach improves the precision of map accuracy assessments and area estimates by ensuring representation from all strata and potentially reducing the uncertainty compared to simple random sampling, especially when the characteristic of interest varies significantly across strata. Essentially, it allows for targeted sampling to capture the variability within the landscape more effectively.

In this notebook an experimental setup based on simulated data is presented to demonstrate the process of stratified random sampling.

In [ ]:
import os
import numpy as np
import pandas as pd
import concurrent.futures
from itertools import product

from pathlib import Path
from matplotlib import pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.lines import Line2D

import seaborn as sns

### 0 - Generic helper functions

These functions will be used throughout the notebook to ease execution.

In [ ]:
# this function allows us to run code on multiple CPUs at once
def run_in_parallel(func, arg_list, workers):

    # initialize the excuter
    executor = concurrent.futures.ProcessPoolExecutor(workers)
    
    # submit tasks
    futures = [executor.submit(func, *args) for args in arg_list]

    # gather results
    results = [
        future.result()
        for future in concurrent.futures.as_completed(futures)
    ]
    executor.shutdown()
    return results
    

# Function to plot 2 arrays side by side
def plot_side_by_side(change_array, stratification, samples=100, allocation='equal'):
    fig, ax = plt.subplots(1, 2, figsize=(20, 10))

    if allocation == 'proportional':
        p = np.sum(change_array)/change_array.size
        s_chg = int(samples * p)
        s_stb = samples - s_chg

    elif allocation == 'equal':
        s_chg = int(samples/2)
        s_stb = samples - s_chg

    if samples:
        idx = np.where(stratification == 1)
        subset = np.random.choice(len(idx[0]), s_chg, replace=False)
        ax[0].plot(idx[1][subset], idx[0][subset], marker='o', color='k', linestyle='none', alpha=0.35, label='Change Samples')
        ax[1].plot(idx[1][subset], idx[0][subset], marker='o', color='k', linestyle='none', alpha=0.35, label='Change Samples')
    
        idx = np.where(stratification == 0)
        subset = np.random.choice(len(idx[0]), s_stb, replace=False)
        ax[0].plot(idx[1][subset], idx[0][subset], marker='o', color='w', linestyle='none', alpha=0.5, label='Stable Samples')
        ax[1].plot(idx[1][subset], idx[0][subset], marker='o', color='w', linestyle='none', alpha=0.5, label='Stable Samples')

    ax[0].imshow(change_array, cmap='RdYlGn_r')
    ax[0].set_title('Actual Change')
    
    ax[1].imshow(stratification, cmap='RdYlGn_r')
    ax[1].set_title('(Noisy) Stratification')
    ax[1].legend()
    
    plt.tight_layout()

## 1 - Create a hypothetical "true map" with a predefined proportion of "change"

Just like in notebook 1, we are defining  a function that allows us creating a simplified representation of a geographical area, much like a digital map. The code accomplishes this by generating a two-dimensional array (think of it as a grid or matrix) filled with random numbers.

A certain percentage of those pixels will be given the value 1 for change, and 0 for no change. The *proportion_of_change* variable will define how many points are going to be 1s.

In [ ]:
# a small helper function to create a fake image of 1s and 0s
def create_true_map(size, proportion_of_change, shuffle=True):

    # create an 2-d array of size squared
    array = np.zeros(size*size)

    # calculate number of change cells
    change_samples = int(size * size * proportion_of_change)
    
    # change the first cells according ot the number of change cells to 1
    array[:change_samples] = 1  # Assign rows to stratum 1
    
    # optionally shuffle array
    if shuffle:
        np.random.shuffle(array)
    
    return array.reshape(size, size)

## 2 - Create a hypothetical stratification with a certain level of noise

In real-world applications, the map used for stratification (e.g., a land cover map showing potential change areas) is never perfectly accurate compared to the actual ground truth. It will contain errors. This function simulates the creation of such an imperfect stratification map based on our "ground truth" map (`true_map`).

We introduce two types of errors, controlled by the `omission_error` and `comission_error` parameters, to mimic inaccuracies often found in thematic maps:

1.  **Omission Error (`omission_error` parameter):** This simulates change that occurred on the ground but was *missed* by the stratification map.
    * The function takes the `true_map` (where '1' represents actual change).
    * It calculates the total number of actual change pixels (`nr_of_change`).
    * A specified proportion (`omission_error`) of these *actual change pixels* are randomly selected.
    * In the resulting `noisy_strata` map, these selected pixels are flipped from '1' (change) to '0' (no change).


2.  **Commission Error (`comission_error` parameter):** This simulates areas that did *not* actually change on the ground but were *incorrectly mapped* as change in the stratification map.
    * The function identifies the pixels that are actually '0' (no change) in the `true_map`.
    * It calculates the *number* of commission errors to introduce. **Note:** In this specific implementation, this number is determined by multiplying the `comission_error` proportion by the *total number of actual change pixels* (`nr_of_change`), `nr_of_comissions = int(nr_of_change * comission_error)`.
    * This calculated number (`nr_of_comissions`) of *actual no-change pixels* are randomly selected.
    * In the resulting `noisy_strata` map, these selected pixels are flipped from '0' (no change) to '1' (change).

The `introduce_noise` function takes the `true_map` and these two error parameters as input and returns the `noisy_strata` array. This array represents the imperfect map layer that will be used to define the strata (e.g., stratum '1' for mapped change, stratum '0' for mapped no change) in the subsequent stratified random sampling process. The level of noise introduced via `omission_error` and `comission_error` will influence how effectively the stratification improves the precision of our final estimate compared to simple random sampling.

In [ ]:
def introduce_noise(true_map, omission_error, comission_error, seed=42):
    """
    Generates 2D arrays representing clean and noisy strata.
    
    Args:
    true_map: A 2d numpy array representing the true distribution of change.
    stratum_1_ratio: The proportion of rows belonging to stratum 1.
    noise_level_0: The percentage of cells to flip in stratum 0.
    noise_level_1: The percentage of cells to flip in stratum 1.
    
    Returns:
    A tuple containing two 2D numpy arrays:
      - clean_strata: The array representing the clean strata.
      - noisy_strata: The array representing the strata with noise.
    """
    
    # Create a copy of the true map to apply noise
    noisy_strata = np.copy(true_map)  
    
    # get the total pupulation size
    population_size = noisy_strata.size
    
    # get the fraction and number of cells of actual change from the true map
    fraction_of_actual_change = np.sum(true_map)/true_map.size
    nr_of_change = fraction_of_actual_change * population_size
    
    # calculate the number of pixels being subject to omission error of stratum 1  
    nr_of_omissions = int(nr_of_change * omission_error)

    # calculate the number of pixels being subject to comission error of strata 1   
    nr_of_comissions = int(nr_of_change * comission_error)
    
    # Separate indices for each stratum
    stratum_0_indices = np.where(noisy_strata == 0)
    stratum_1_indices = np.where(noisy_strata == 1)
    
    # Introduce noise in stratum 0
    indices_to_flip_0 = np.random.choice(len(stratum_0_indices[0]), nr_of_comissions, replace=False)
    noisy_strata[stratum_0_indices[0][indices_to_flip_0], stratum_0_indices[1][indices_to_flip_0]] = 1
    
    # Introduce noise in stratum 1
    indices_to_flip_1 = np.random.choice(len(stratum_1_indices[0]), nr_of_omissions, replace=False)
    noisy_strata[stratum_1_indices[0][indices_to_flip_1], stratum_1_indices[1][indices_to_flip_1]] = 0
    
    return noisy_strata

### Example: Generating and Visualizing True Map and Noisy Stratification

This code block demonstrates the process of creating a simulated "ground truth" map and then generating a corresponding "noisy" stratification map that includes errors, mimicking a real-world scenario. Finally, it visualizes both maps.

1.  **Parameter Setup:**
    * `size = 100`: Sets the dimension for our square map, resulting in a 100x100 grid (10,000 total pixels).
    * `fraction_of_actual_change = 0.01`: Defines the true proportion of "change" (value '1') in the ground truth map. In this 100x100 map, this corresponds to $10000 \times 0.01 = 100$ actual change pixels.
    * `omission_error = 0.1`: Specifies that 10% of the actual change pixels should be incorrectly labeled as "no change" (value '0') in the stratification map. This means $100 \times 0.1 = 10$ true '1's will be flipped to '0'.
    * `comission_error = 0.2`: Specifies the rate for commission errors. Based on the `introduce_noise` function's logic, the *number* of "no change" pixels (value '0') incorrectly labeled as "change" (value '1') in the stratification map is calculated as 20% of the *total number of actual change pixels*. This means $100 \times 0.2 = 20$ true '0's will be flipped to '1'.

2.  **Map Creation:**
    * `true_map = create_true_map(size, fraction_of_actual_change, shuffle=False)`: This line calls the function to generate the 100x100 `true_map`. It contains exactly 100 pixels with the value '1' (representing change) and the rest '0'. Since `shuffle=False`, the '1' pixels are likely initially grouped together in the array.
    * `stratification = introduce_noise(true_map, omission_error, comission_error)`: This line calls the `introduce_noise` function. It takes the `true_map` and applies the specified errors: it flips 10 true '1's to '0' (omission) and 20 true '0's to '1' (commission). The result is stored in the `stratification` variable, representing an imperfect map layer derived from the ground truth.

3.  **Visualization:**
    * `plot_side_by_side(true_map, stratification, 100)`: This line uses the helper function to display the generated `true_map` and the `stratification` map next to each other. This allows for a visual comparison, showing where the noisy stratification correctly identifies change, misses change (omission), or incorrectly identifies change (commission). The `samples=100` argument also overlays 100 sample points onto the plots (likely 50 in the area mapped as change and 50 in the area mapped as stable in the `stratification` map, based on the default 'equal' allocation) to illustrate how samples might be drawn based on the stratification.

In [ ]:
size = 100
fraction_of_actual_change = 0.01
omission_error = 0.1
comission_error = 0.2

true_map = create_true_map(size, fraction_of_actual_change, shuffle=False)
stratification = introduce_noise(true_map, omission_error, comission_error)

plot_side_by_side(true_map, stratification, 100)

## 3 - Stratified Random Sampling Estimator: Calculation Details

The core idea behind the stratified random sampling estimator is to divide the population into distinct groups (strata) based on some known characteristic (represented by our `stratification` map) and then perform sampling independently within each stratum. The results from each stratum are then combined, weighted by the stratum's size, to produce an overall estimate for the population parameter (in this case, the proportion of "change"). This approach often leads to more precise estimates (lower standard error) compared to simple random sampling, especially if the strata effectively group areas with different characteristics.

The process, as implemented in the `stratified_random_sampling` function, involves two main stages:

1.  **Per-Stratum Estimation:**
    * The code first identifies the unique strata (e.g., stratum '0' for mapped no change, stratum '1' for mapped change) present in the input `stratification` array.
    * For each stratum $h$, it determines:
        * The total number of population units belonging to that stratum ($N_h$) based on the `stratification` map.
        * The stratum weight ($W_h$), calculated as the proportion of the total population ($N$) that falls into stratum $h$: $W_h = N_h / N$.
        * The required sample size for the stratum ($n_h$), determined based on the chosen `allocation_strategy` ('proportional' to $W_h$ or 'equal' across strata) and the `target_sample_size`.
    * The function `per_stratum_estimation` is then called for each stratum $h$. This function performs the actual sampling *within* that stratum:
        * It draws $n_h$ samples randomly *without replacement* from the `true_map` pixels corresponding to stratum $h$.
        * It calculates the sample proportion ($p_{\hat{h}}$) for stratum $h$ based on the sampled values from the `true_map`.
        * Crucially, it also calculates the standard error ($se_h$) for this within-stratum estimate ($p_{\hat{h}}$). This calculation mirrors the standard error formula used in simple random sampling but is applied only to the stratum's data. As seen in the `per_stratum_estimation` code, it uses the formula $se_h = \sqrt{(1 - \frac{n_h}{N_h}) \frac{p_{\hat{h}}(1-p_{\hat{h}})}{n_h-1}}$, incorporating both the Finite Population Correction ($1 - n_h/N_h$) and Bessel's correction ($n_h-1$ denominator).

2.  **Combining Stratum Estimates:**
    * After obtaining the estimates ($p_{\hat{h}}$) and their standard errors ($se_h$) for each stratum, the `stratified_random_sampling` function combines them to get the overall stratified estimate and its standard error.
    * **Overall Proportion ($p_{\hat{st}}$):** The overall estimated proportion is calculated as the weighted sum of the individual stratum proportions:
        $$ p_{\hat{st}} = \sum_{h} W_h p_{\hat{h}} $$
        In the code, this is achieved by summing the `proportion` column of the `est_df` DataFrame, where each entry already represents the weighted proportion $W_h \times p_{\hat{h}}$.
    * **Overall Standard Error ($SE_{\hat{p}_{st}}$):** The standard error of the overall stratified estimate is calculated by combining the standard errors from each stratum. The specific formula implemented in the code is:
        $$ SE_{\hat{p}_{st}} = \sqrt{\sum_{h} (W_h se_h)^2} $$
        This is calculated in the code by taking the square root of the sum of the squares of the `se` column in the `est_df` DataFrame, where each entry represents the weighted standard error $W_h \times se_h$.

This calculated overall proportion ($p_{\hat{st}}$) and its standard error ($SE_{\hat{p}_{st}}$) represent the final result of the stratified random sampling estimation for a single run.

In [ ]:
def per_stratum_estimation(true_map, indices, N_h, n_h, seed=42):

    # set seed
    np.random.seed(seed=seed)

    # select indices
    sampled_indices = np.random.choice(
        N_h,
        n_h,  
        replace=False
    )

    # sample data
    sampled = true_map[indices[0][sampled_indices]]

    # calculate proportion
    p_hat = np.sum(sampled)/n_h
    
    # calculate variance
    S = p_hat * (1-p_hat)

    # calculate finite element correction
    finite = 1 - n_h/N_h

    # calculate the standard error
    se = np.sqrt(finite * S/(n_h - 1))
    
    return p_hat, se, np.sum(sampled)


def stratified_random_sampling(stratification, true_map, target_sample_size, allocation_strategy, seed=42):
    """
    Performs stratified random sampling on a population.
    
    Args:
    stratification: A 2D array representing the strata.
    true_map: A 2D array representing the true population that is going to be sampled.
    sample_size_1: The desired sample size from stratum 1.
    sample_size_2: The desired sample size from stratum 2.
    
    Returns:
    A list of sampled values.
    """

    stratification = stratification.flatten()
    true_map = true_map.flatten()
    
    # get stratum numbers
    strata = np.unique(stratification)
    N = stratification.size

    # actual sampling
    estimates = {}
    for h in sorted(strata):

        # get array indices of stratum h
        indices = np.where(stratification == h)

        # get stratum size
        N_h = len(indices[0])
        
        # get stratum weight
        W_h = N_h/N
        
        # get proportional sample size for stratum h
        if allocation_strategy == 'proportional':
            n_h = int(W_h*target_sample_size)

        # get equal sample size for stratum h
        if allocation_strategy == 'equal':
            n_h = int(target_sample_size/len(strata))
        
        # calculate the per stratum proportion and SE
        p_hat_h, se, _sum = per_stratum_estimation(true_map, indices, N_h, n_h, seed=seed)

        # store in dict
        estimates[h] = [n_h, _sum, W_h*p_hat_h, W_h*se]

    est_df = pd.DataFrame.from_dict(estimates, orient='index', columns=['samples', 'sum', 'proportion', 'se'])
    display(est_df)
    proportion = est_df.proportion.sum()
    se = np.sqrt(np.sum([se**2 for se in est_df.se.values]))
    return target_sample_size, proportion, se, 100 * 1.645 * se / proportion, allocation_strategy
    

def simple_random_sampling(true_map, sample_size, seed=42):

    # set random seed according to given seed
    np.random.seed(seed=seed)

    N_h = true_map.size 
    n_h = sample_size
    
    # randomly sample the given array with the given sample size
    sampled = np.random.choice(true_map.flatten(), n_h, replace=False)

    # calculate the proportion
    p_hat = sampled.sum()/len(sampled)
    
    # calculate variance S
    S = p_hat * (1-p_hat)

    # calculate finite element correction
    finite = 1 - n_h/N_h

    # calculate the standard error
    se = np.sqrt(finite * S/(n_h - 1))

    return sample_size, p_hat, se, 100 * 1.645 * se / p_hat, 'simple'

## 4 - Simulation

As in notebook #1, we are running both, a simple and a stratified random sampling, over a number of `iterations` with fixed `sample size`. With this, we can proof the unbiasedness of the stratified random estimator, and demonstrate its efficiency for reasonable omission and comission error. We therefore use the map from the example code above.

In [ ]:
# Create True Population and Map
size = 1000
fraction_of_actual_change = 0.01
omission_error = 0.1
comission_error = 0.2

true_map = create_true_map(size, fraction_of_actual_change, shuffle=False)
stratification = introduce_noise(true_map, omission_error, comission_error)

# Set sample size and nr. of iteration
sample_size = 3000
iterations = 1000

# Run Simple Random Sampling
args_list = [(true_map, sample_size, i) for i in range(iterations)]
results = run_in_parallel(simple_random_sampling, args_list, int(os.cpu_count()/2))
srs_results = pd.DataFrame(results, columns=['Sample Size', 'p_hat', 'SE', '90% CI', 'Allocation Scheme'])

# Run Stratified Random Sampling
args_list = []
for alloc in ['equal', 'proportional']:
        args_list.extend((stratification, true_map, sample_size, alloc, i) for i in range(iterations))
        
results = run_in_parallel(stratified_random_sampling, args_list, int(os.cpu_count()/2))
strrs_results = pd.DataFrame(results, columns=['Sample Size', 'p_hat', 'SE', '90% CI', 'Allocation Scheme'])

results = pd.concat([srs_results, strrs_results])

### Plot the results

In [ ]:
# plot the distribution of the Estimates as a Violinplot, grouped by sample size
fig, ax = plt.subplots(1, 2, figsize=(15,8))

# first plot ( as above) with the distribution of estimates and true value
ax[0] = sns.violinplot(results, y='p_hat', x='Allocation Scheme', hue='Allocation Scheme', ax=ax[0], palette='magma', legend=False)
ax[0].axhline(fraction_of_actual_change, c='red', linestyle=':')
ax[0].set_title('Distribution of the estimated proportion for different sampling sizes')

# Add legend for true value
handles, labels = ax[0].get_legend_handles_labels()
point = Line2D([0], [0], label='True value', linestyle=':', c='r')
handles.append(point) 
ax[0].legend(handles=handles)

# second plot with distribution of estimated SE's and calculated one (red dots)
ax[1] = sns.violinplot(results, y='SE', x='Allocation Scheme', hue='Allocation Scheme', ax=ax[1], palette='magma', legend=False)
ax[1] = sns.pointplot(results.groupby('Allocation Scheme').std().reset_index(), x='Allocation Scheme', y='p_hat', ax=ax[1], c='red', markersize=1.5, linestyle=' ')
ax[1].set_title('Distribution of the estimated SE for different sampling sizes')

# Add legend for red dots
handles, labels = ax[1].get_legend_handles_labels()
point = Line2D([0], [0], label='Calculated SE', marker='.', markersize=10, markerfacecolor='r', markeredgecolor='r', linestyle='')
handles.append(point) 
ax[1].legend(handles=handles)

plt.tight_layout()
plt.show()

## Lessons Learned and Takeaways

This notebook provided a practical demonstration of Stratified Random Sampling (STRS) using simulated data, highlighting its mechanics and comparing it to Simple Random Sampling (SRS). Here are the key takeaways:

* **Stratification Purpose:** STRS divides a population into distinct subgroups (strata) based on auxiliary information (like a map) before sampling. The goal is often to improve the precision (reduce the standard error) of the overall estimate compared to SRS, especially when the characteristic being measured (e.g., "change") varies significantly between strata.
* **Simulating Reality:** Real-world maps used for stratification are imperfect. The notebook simulated this by introducing "noise" (omission and commission errors) into a stratification layer derived from a known "true map". The quality (accuracy) of the stratification map impacts the effectiveness of STRS.
* **Stratified Estimator Mechanics:**
    * Samples are drawn independently from each stratum.
    * Estimates (e.g., proportion $p_{\hat{h}}$) and their standard errors ($se_h$) are calculated within each stratum.
    * The overall estimate ($p_{\hat{st}}$) is a *weighted* average of the stratum estimates, where weights ($W_h$) are based on the relative size (area/pixel count) of each stratum in the population: $p_{\hat{st}} = \sum W_h p_{\hat{h}}$.
    * The overall standard error ($SE_{\hat{p}_{st}}$) is calculated by combining the within-stratum standard errors, also considering the weights: $SE_{\hat{p}_{st}} = \sqrt{\sum (W_h se_h)^2}$.
* **Unbiasedness:** Like SRS, the STRS estimator is unbiased. The simulations demonstrated that the average of many STRS estimates converges to the true population proportion.
* **Potential for Increased Precision:** The simulations showed that STRS (with both equal and proportional allocation in this specific example) resulted in a lower standard error compared to SRS for the same total sample size. This demonstrates the potential efficiency gain from using stratification, provided the stratification is reasonably effective at separating areas with different characteristics.
* **Allocation Strategy:** How the total sample size is allocated among strata matters. The notebook implemented 'proportional' (sample size $n_h$ proportional to stratum size $N_h$) and 'equal' allocation ($n_h$ is the same for all strata). While both stratified methods outperformed SRS in the simulation, different allocation strategies (including optimal allocation, not shown here) can yield different levels of precision. Equal allocation can be particularly useful for ensuring sufficient samples in rare strata.
* **Impact of Stratification Errors:** While not explicitly varied in the final simulation, the introduction of noise (omission/commission errors) highlights that the *effectiveness* of STRS relies on the quality of the stratification layer. High error rates in the stratification map reduce the potential gains in precision compared to SRS.